## Notebook 2 — Feature Engineering

This notebook builds the feature matrix used to train our ML models.

The cohort of 13,768 funded companies (founded 1995–2009) is loaded from
the processed data folder.

In [ ]:
#data loading 
import pandas as pd
import numpy as np
from pathlib import Path

# Paths
DATA_RAW       = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(exist_ok=True)

np.random.seed(42)
# Load cohort from notebook 1
cohort = pd.read_csv(DATA_PROCESSED / 'cohort.csv', low_memory=False)

# Load supporting tables
rounds  = pd.read_csv(DATA_RAW / 'funding_rounds.csv', low_memory=False)
inv     = pd.read_csv(DATA_RAW / 'investments.csv',    low_memory=False)
acq     = pd.read_csv(DATA_RAW / 'acquisitions.csv',   low_memory=False)
ipos    = pd.read_csv(DATA_RAW / 'ipos.csv',           low_memory=False)

print("Cohort loaded:", cohort.shape)
print("Funding rounds:", rounds.shape)
print("Investments:", inv.shape)
print("Label distribution:")
print(cohort['label'].value_counts())

Cohort loaded: (13768, 42)
Funding rounds: (52928, 23)
Investments: (80902, 6)
Label distribution:
label
0    11850
1     1918
Name: count, dtype: int64


### Feature Group 1 — Funding history

 We extract five core features from the funding_rounds table:

- **n_rounds**: total number of funding rounds : more rounds signal investor confidence
- **total_funding_usd**: total capital raised : log-transformed to handle skewness
- **avg_round_size**: average round size : proxy for valuation trajectory
- **funding_span_months**: time between first and last round : longer runway suggests sustained growth
- **log_total_funding / log_avg_round**: log transforms of the above to reduce skewness

We group by company ID and aggregate across all rounds recorded before the snapshot date.

In [3]:
# Parse dates
rounds['funded_at'] = pd.to_datetime(rounds['funded_at'], errors='coerce')

# Group by company
funding_features = rounds.groupby('object_id').agg(
    n_rounds          = ('funding_round_id', 'count'),
    total_funding_usd = ('raised_amount_usd', 'sum'),
    avg_round_size    = ('raised_amount_usd', 'mean'),
    first_funding     = ('funded_at', 'min'),
    last_funding      = ('funded_at', 'max'),
).reset_index()

# Time between first and last round (months)
funding_features['funding_span_months'] = (
    (funding_features['last_funding'] - funding_features['first_funding'])
    .dt.days / 30
).round(1)

# Log transform skewed funding amounts
funding_features['log_total_funding'] = np.log1p(
    funding_features['total_funding_usd'].fillna(0))
funding_features['log_avg_round']     = np.log1p(
    funding_features['avg_round_size'].fillna(0))
funding_features['funding_span_months'] = funding_features[
    'funding_span_months'].clip(upper=120)  # max 10 years

print("Funding features shape:", funding_features.shape)
print(funding_features[['n_rounds','total_funding_usd',
                         'funding_span_months']].describe().round(1))

Funding features shape: (31939, 9)
       n_rounds  total_funding_usd  funding_span_months
count   31939.0       3.193900e+04              31735.0
mean        1.7       1.316794e+07                 10.0
std         1.2       6.729957e+07                 19.5
min         1.0       0.000000e+00                  0.0
25%         1.0       2.000000e+05                  0.0
50%         1.0       1.700000e+06                  0.0
75%         2.0       9.100000e+06                 13.0
max        15.0       5.700000e+09                120.0


### Feature Group 2 — Funding round types

The type of funding round reached is a strong signal of company maturity 
and investor confidence. A company that has reached Series B is fundamentally 
different from one that only raised a seed round. We create binary indicators 
for each round type, plus a feature capturing the highest stage reached.

In [ ]:

round_type_dummies = pd.get_dummies(
    rounds[['object_id', 'funding_round_type']].dropna(),
    columns=['funding_round_type']
)

round_type_feats = round_type_dummies.groupby('object_id').max().reset_index()

print("Round type features shape:", round_type_feats.shape)
print("\nRound type columns:")
print([c for c in round_type_feats.columns if c != 'object_id'])

Round type features shape: (31939, 10)

Round type columns:
['funding_round_type_angel', 'funding_round_type_crowdfunding', 'funding_round_type_other', 'funding_round_type_post-ipo', 'funding_round_type_private-equity', 'funding_round_type_series-a', 'funding_round_type_series-b', 'funding_round_type_series-c+', 'funding_round_type_venture']


### Feature Group 3 — Company characteristics

Company level features from the objects table. These capture sector, 
geography, company age, and network signals. Country and sector are encoded 
as binary indicators (US vs non-US, top sectors vs other) to avoid high 
cardinality issues.

**Feature justification:**
- **is_us**: US companies have significantly higher exit rates due to deeper 
  M&A markets and IPO pipelines (Arroyo et al., 2019). Binary encoding 
  avoids high-cardinality country dummies.
- **company_age**: Age at snapshot (2014 minus founding year) controls for 
  the fact that older companies had more time to exit within the observation 
  window. Without this feature, the model would confound "hasn't exited" 
  with "hasn't had time to exit".
- **relationships and milestones**: Signals for company activity and 
  network reach within the Crunchbase ecosystem.

In [10]:

cohort['founded_year'] = pd.to_datetime(
    cohort['founded_at'], errors='coerce').dt.year

# Company age at snapshot (2014)
cohort['company_age'] = 2014 - cohort['founded_year']

# US company binary flag
cohort['is_us'] = (cohort['country_code'] == 'USA').astype(int)

# Top sectors as binary flags (sectors with >100 companies)
top_sectors = cohort['category_code'].value_counts()
top_sectors = top_sectors[top_sectors >= 100].index.tolist()
print(f"Top sectors (>=100 companies): {len(top_sectors)}")
print(top_sectors)

# Create sector dummies for top sectors only
sector_dummies = pd.get_dummies(
    cohort['category_code'].apply(
        lambda x: x if x in top_sectors else 'other'
    ), prefix='sector'
)

# Network signals from objects table
cohort['relationships'] = cohort['relationships'].fillna(0)
cohort['milestones']    = cohort['milestones'].fillna(0)

print(f"\nCompany age range: {cohort['company_age'].min()} to {cohort['company_age'].max()} years")
print(f"US companies: {cohort['is_us'].sum():,} ({cohort['is_us'].mean()*100:.1f}%)")

Top sectors (>=100 companies): 24
['software', 'biotech', 'web', 'enterprise', 'mobile', 'games_video', 'advertising', 'ecommerce', 'cleantech', 'hardware', 'medical', 'network_hosting', 'semiconductor', 'other', 'analytics', 'public_relations', 'security', 'health', 'finance', 'search', 'consulting', 'education', 'manufacturing', 'social']

Company age range: 5 to 19 years
US companies: 9,406 (68.3%)


### Feature Group 4 — Investor signals

The number and type of investors signals deal quality. Companies backed by 
more investors have broader networks and more exit opportunities. So I counted 
unique investors per company from the investments table.

In [11]:

investor_features = inv.groupby('funded_object_id').agg(
    n_investors = ('investor_object_id', 'nunique')
).reset_index().rename(columns={'funded_object_id': 'object_id'})

print("Investor features shape:", investor_features.shape)
print(investor_features['n_investors'].describe().round(1))

Investor features shape: (21607, 2)
count    21607.0
mean         3.0
std          2.9
min          1.0
25%          1.0
50%          2.0
75%          4.0
max         49.0
Name: n_investors, dtype: float64


**Feature Group 5 — Milestones** (source: milestones.csv)  
Milestones capture company traction and visibility. Each milestone represents 
a notable event : product launches, partnerships, user growth targets. 

In [23]:

# object_id links to our cohort companies
# n_milestones = total number of notable events recorded

milestone_feats = milestones.groupby('object_id').agg(
    n_milestones = ('id', 'count')   # count rows per company
).reset_index()

print("=== Feature Group 5: Milestones ===")
print(f"Companies with milestones: {len(milestone_feats):,}")
print(milestone_feats['n_milestones'].describe().round(1))

=== Feature Group 5: Milestones ===
Companies with milestones: 17,159
count    17159.0
mean         2.3
std          3.4
min          1.0
25%          1.0
50%          1.0
75%          2.0
max         75.0
Name: n_milestones, dtype: float64


**Feature Group 6 — Offices** (source: offices.csv)  
Office count proxies for organisational scale and geographic expansion. 
A company operating across multiple cities or countries has crossed 
significant operational thresholds. I flagged whether any office 
is located outside the company's home country, as international presence 
signals global ambition and increases acquisition attractiveness to 
multinational buyers (Arroyo et al., 2019).

In [27]:
# object_id = links each office to its company
# office_id = unique ID for each office location
# country_code = country where this office is located

# Step 1: Count total offices per company
# groupby object_id = group all offices belonging to same company
# count office_id = how many office rows exist per company
# result: one row per company with n_offices column
office_feats = offices.groupby('object_id').agg(
    n_offices = ('office_id', 'count')
).reset_index()

# Step 2: Get each company's home country from cohort
home_country = cohort[['id', 'country_code']].rename(
    columns={'id': 'object_id'})# rename 'id' to 'object_id' so we can merge on the same column name

# Step 3: Merge offices with home country

offices_merged = offices.merge(home_country, on='object_id', how='left')# country_code_x = this office's country
# country_code_y = company's home country

# Step 4: Flag international offices
# An office is international if:
#   (a) the office country is DIFFERENT from the home country AND (b) BOTH countries are known (not missing)
# Avoids false positives from missing country codes
offices_merged['is_international'] = (
    (offices_merged['country_code_x'] !=
     offices_merged['country_code_y']) & # different country
    (offices_merged['country_code_x'].notna()) & # office country known
    (offices_merged['country_code_y'].notna())  # home country known
).astype(int) # convert True/False to 1/0

# Step 5: Aggregate to company level
# .max() on binary = 1 if company had ANY international office
intl_feats = offices_merged.groupby('object_id').agg(
    has_international_office = ('is_international', 'max')
).reset_index()

# Step 6: Combine office count + international flag
office_feats = office_feats.merge(intl_feats, on='object_id', how='left')
office_feats['has_international_office'] = (
    office_feats['has_international_office'].fillna(0)) # fillna(0) = companies not in intl_feats had no international office

# ── Final output ──────────────────────────────────────────────
print("=== Feature Group 6: Offices ===")
print(f"Shape: {office_feats.shape}")
print(f"\nn_offices:")
print(office_feats['n_offices'].describe().round(1))
print(f"\nhas_international_office:")
print(f"  Yes: {office_feats['has_international_office'].sum():.0f}")
print(f"  No:  {(office_feats['has_international_office']==0).sum():.0f}")

=== Feature Group 6: Offices ===
Shape: (95043, 3)

n_offices:
count    95043.0
mean         1.2
std          0.8
min          1.0
25%          1.0
50%          1.0
75%          1.0
max         81.0
Name: n_offices, dtype: float64

has_international_office:
  Yes: 1084
  No:  93959


**Feature Group 7 — Team & Leadership** (source: relationships.csv)  
The relationships table captures all people associated with each company. 
I distinguished between total historical team size (n_team_total) and current 
active members (n_team_current, where is_past=0). Current team size reflects 
operational capacity at the snapshot date, while total team size captures 
cumulative human capital investment.

In [25]:
# relationships.csv: one row per person-company association
# relationship_object_id = the company
# is_past = 0 means currently active, 1 means former
# n_team_total   = all people ever associated (historical depth)
# n_team_current = only current active members (operational capacity)
# ═══════════════════════════════════════════════════════════════

team_feats = relationships.groupby('relationship_object_id').agg(
    # count ALL rows = total people ever associated
    n_team_total   = ('id', 'count'),

    # count only current members (is_past == 0)
    # lambda applies a custom function to each group
    # (x == 0) creates True/False per row, .sum() counts Trues
    n_team_current = ('is_past', lambda x: (x == 0).sum())

).reset_index().rename(
    # rename to object_id so it matches all other tables for merging
    columns={'relationship_object_id': 'object_id'})

print("\n=== Feature Group 7: Team ===")
print(f"Companies with team data: {len(team_feats):,}")
print(team_feats[['n_team_total', 'n_team_current']].describe().round(1))

# ═══════════════════════════════════════════════════════════════
# SANITY CHECK — all shapes correct before merging
# ═══════════════════════════════════════════════════════════════
print("\n=== Sanity check ===")
print(f"Cohort:           {len(cohort):,} companies")
print(f"Milestone feats:  {len(milestone_feats):,} companies")
print(f"Office feats:     {len(office_feats):,} companies")
print(f"Team feats:       {len(team_feats):,} companies")
print(f"Note: not all cohort companies appear in every table")
print(f"Missing values will be filled with 0 after merging")


=== Feature Group 7: Team ===
Companies with team data: 137,289
       n_team_total  n_team_current
count      137289.0        137289.0
mean            2.9             1.8
std             9.3             3.5
min             1.0             0.0
25%             1.0             0.0
50%             1.0             1.0
75%             3.0             2.0
max          1178.0           317.0

=== Sanity check ===
Cohort:           13,768 companies
Milestone feats:  17,159 companies
Office feats:     95,043 companies
Team feats:       137,289 companies
Note: not all cohort companies appear in every table
Missing values will be filled with 0 after merging


### Feature Group 8 — Education

Educational pedigree of the founding/executive team is operationalised as a 
binary indicator using an external ranking definition, avoiding circular 
feature definition from the target variable.

Top institutions are defined as:
- US News top 25 universities 2010
- Oxbridge (Oxford, Cambridge)
- Top international institutions (INSEAD, IIT, LSE, Imperial)

This is consistent with Beckman & Burton (2008), who define 
elite education using published external rankings rather than sample-derived 
criteria.

In [20]:


top_unis = [
    # US News Top 25 National Universities (2010)
    'harvard', 'princeton', 'yale', 'columbia',
    'penn', 'wharton', 'duke', 'mit', 'stanford',
    'caltech', 'dartmouth', 'northwestern',
    'johns hopkins', 'washington university',
    'brown', 'cornell', 'notre dame', 'vanderbilt',
    'rice', 'georgetown', 'emory', 'carnegie mellon',
    'virginia', 'ucla', 'michigan', 'berkeley',
    # UK elite (equivalent tier)
    'oxford', 'cambridge',
    'london school of economics', 'lse',
    'imperial college',
    'london business school',
    # Top international business schools
    'insead',
    # Indian Institutes of Technology (elite technical)
    'iit',
]
# Rebuild education features with proper university list
deg_linked['has_mba'] = deg_linked['degree_type'].str.contains(
    'MBA|M.B.A', case=False, na=False).astype(int)

deg_linked['has_phd'] = deg_linked['degree_type'].str.contains(
    'PhD|Ph.D|Doctor', case=False, na=False).astype(int)

# Match against external ranking list
deg_linked['has_top_uni'] = deg_linked['institution'].str.lower().str.contains(
    '|'.join(top_unis), na=False).astype(int)

# Aggregate to company level
edu_feats = deg_linked.groupby('object_id_y').agg(
    has_mba     = ('has_mba',     'max'),
    has_phd     = ('has_phd',     'max'),
    has_top_uni = ('has_top_uni', 'max')
).reset_index().rename(columns={'object_id_y': 'object_id'})

print("Education features shape:", edu_feats.shape)
print(f"\nHas MBA:     {edu_feats['has_mba'].sum():,}")
print(f"Has PhD:     {edu_feats['has_phd'].sum():,}")
print(f"Has top uni: {edu_feats['has_top_uni'].sum():,}")

# Validate — do top-uni companies exit more?
cohort_edu = cohort.merge(
    edu_feats, left_on='id', right_on='object_id', how='left')
cohort_edu['has_top_uni'] = cohort_edu['has_top_uni'].fillna(0)
cohort_edu['has_mba']     = cohort_edu['has_mba'].fillna(0)
cohort_edu['has_phd']     = cohort_edu['has_phd'].fillna(0)

exit_top    = cohort_edu[cohort_edu['has_top_uni']==1]['label'].mean()*100
exit_no_top = cohort_edu[cohort_edu['has_top_uni']==0]['label'].mean()*100
exit_mba    = cohort_edu[cohort_edu['has_mba']==1]['label'].mean()*100
exit_phd    = cohort_edu[cohort_edu['has_phd']==1]['label'].mean()*100

print(f"\n=== Validation: exit rates by education ===")
print(f"Cohort average:          {cohort['label'].mean()*100:.1f}%")
print(f"Has top uni:             {exit_top:.1f}%")
print(f"No top uni:              {exit_no_top:.1f}%")
print(f"Has MBA:                 {exit_mba:.1f}%")
print(f"Has PhD:                 {exit_phd:.1f}%")
print(f"Top uni lift:            {exit_top/exit_no_top:.2f}x")

Education features shape: (87043, 4)

Has MBA:     30,294
Has PhD:     10,301
Has top uni: 39,258

=== Validation: exit rates by education ===
Cohort average:          13.9%
Has top uni:             21.4%
No top uni:              8.4%
Has MBA:                 21.7%
Has PhD:                 20.0%
Top uni lift:            2.56x


### Feature Groups 9 — Funding recency and velocity

Source: funding_rounds.csv

These features capture **when** and **how aggressively** a company has been 
raising money relative to the snapshot date (January 2014).

**Features:**

- **days_since_last_funding**: days between most recent round and snapshot 
  date. Low = recently active. High = possibly dormant. 

- **avg_days_between_rounds**: average gap between consecutive rounds. 
  Companies raising frequently show sustained momentum. Zero for companies 
  with only one round (no gap to calculate).

- **round_size_growth**: last round size divided by first round size. 
  Values >1 = rounds getting larger (healthy trajectory). Values <1 = 
  shrinking rounds (warning sign). Capped at 100x to remove outliers.

- **last_round_size_usd**: dollar amount of most recent round. Larger 
  last rounds indicate later-stage companies closer to exit.

- **log_last_round_size**: log(1 + last_round_size_usd). Log-transformed 
  to handle heavy-tailed distribution of funding amounts, consistent with 
  the transformation applied to total funding in Feature Group 1.


In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
import pandas as pd
import numpy as np

# ═══════════════════════════════════════════════════════════════
# FEATURE GROUP 9 — Funding recency and velocity
# ═══════════════════════════════════════════════════════════════

SNAPSHOT = pd.Timestamp('2014-01-01')
rounds['funded_at'] = pd.to_datetime(rounds['funded_at'], errors='coerce')

# Last round date per company
last_round_date = (rounds.groupby('object_id')['funded_at']
                         .max().reset_index()
                         .rename(columns={'funded_at':'last_round_date'}))

# First round date per company  
first_round_date = (rounds.groupby('object_id')['funded_at']
                          .min().reset_index()
                          .rename(columns={'funded_at':'first_round_date'}))

# Last round size
last_round_size = (rounds.sort_values('funded_at')
                         .groupby('object_id')['raised_amount_usd']
                         .last().reset_index()
                         .rename(columns={'raised_amount_usd':'last_round_size_usd'}))

# First round size
first_round_size = (rounds.sort_values('funded_at')
                          .groupby('object_id')['raised_amount_usd']
                          .first().reset_index()
                          .rename(columns={'raised_amount_usd':'first_round_size_usd'}))

# Merge all recency features
recency = last_round_date.merge(first_round_date, on='object_id')
recency = recency.merge(last_round_size,  on='object_id', how='left')
recency = recency.merge(first_round_size, on='object_id', how='left')

# Days since last funding at snapshot
recency['days_since_last_funding'] = (
    SNAPSHOT - recency['last_round_date']).dt.days

# Average days between rounds
recency['avg_days_between_rounds'] = (
    (recency['last_round_date'] - recency['first_round_date']).dt.days /
    funding_features.set_index('object_id')['n_rounds']
     .reindex(recency['object_id'].values).values
).clip(lower=0)

# Round size growth (last / first)
recency['round_size_growth'] = (
    recency['last_round_size_usd'] /
    recency['first_round_size_usd'].replace(0, np.nan)
).clip(upper=100).fillna(1)

# Log transforms
recency['log_last_round_size'] = np.log1p(
    recency['last_round_size_usd'].fillna(0))

# Keep only needed columns
recency_feats = recency[['object_id', 'days_since_last_funding',
                          'avg_days_between_rounds', 'round_size_growth',
                          'last_round_size_usd', 'log_last_round_size']]

print("=== Feature Group 9: Recency ===")
print(f"Shape: {recency_feats.shape}")
print(recency_feats.describe().round(1))

=== Feature Group 9: Recency ===
Shape: (31939, 6)
       days_since_last_funding  avg_days_between_rounds  round_size_growth  \
count                  31735.0                  31735.0            31939.0   
mean                     999.4                    103.7                2.4   
std                      939.4                    201.0                7.6   
min                       20.0                      0.0                0.0   
25%                      290.0                      0.0                1.0   
50%                      731.0                      0.0                1.0   
75%                     1457.0                    166.0                1.0   
max                    19724.0                   8643.5              100.0   

       last_round_size_usd  log_last_round_size  
count         3.193900e+04              31939.0  
mean          7.716816e+06                 12.3  
std           4.709022e+07                  5.2  
min           0.000000e+00                  0.

### Feature Group 10 — Round Progression and Time Signals

Source: funding_rounds.csv + objects.csv

These features capture **how far a company progressed through the VC funding 
ladder** and **when it was founded** relative to market conditions.

The VC funding ladder represents increasing company maturity:
Angel → Venture → Series A → Series B → Series C+ → Exit


**Binary stage flags** (1=reached, 0=not reached):
- **reached_angel/venture/series_a/series_b/series_c+**: each stage 
  reached is a viability signal. Only ~20-30% of angel companies ever 
  reach Series A; Series C+ companies are prime acquisition targets.

**Summary score:**
- **funding_stage_score**: ordinal 1–5 for highest stage reached. 
  Provides a single continuous maturity signal alongside the binary flags.

**Time signals:**
- **founded_in_recession**: founded in 2008–2009. Recession survivors 
  tend to be more capital-efficient (30% of cohort).
- **first_funding_age**: company age at first round. Faster fundraising 
  signals stronger early investor demand. 


In [52]:
# Which round types did each company reach
rounds_cohort = rounds[rounds['object_id'].isin(cohort['id'])]

# Binary flags for reaching each stage
# Note: series-c+ becomes reached_series_c_plus (+ → _plus, - → _)
for stage in ['angel', 'series-a', 'series-b', 'series-c+', 'venture']:
    col = f"reached_{stage.replace('-','_').replace('+','_plus')}"
    reached = (rounds_cohort[rounds_cohort['funding_round_type']==stage]
                             ['object_id'].unique())
    cohort[col] = cohort['id'].isin(reached).astype(int)

# Funding stage score: highest stage reached
# angel/none=1, venture=2, series-a=3, series-b=4, series-c+=5
def stage_score(row):
    if row['reached_series_c_plus']: return 5  # single underscore
    if row['reached_series_b']:      return 4
    if row['reached_series_a']:      return 3
    if row['reached_venture']:       return 2
    return 1

cohort['funding_stage_score'] = cohort.apply(stage_score, axis=1)

# Time signals
cohort['founded_in_recession'] = cohort['founded_year'].isin(
    [2008, 2009]).astype(int)

cohort['first_funding_age'] = (
    pd.to_datetime(cohort['first_funding_at'], errors='coerce').dt.year -
    cohort['founded_year']
).clip(lower=0, upper=20).fillna(cohort['company_age'].median())

# Build progression features table
progression_cols = ['id', 'reached_angel',
                    'reached_series_a', 'reached_series_b',
                    'reached_series_c_plus', 'reached_venture',
                    'funding_stage_score', 'founded_in_recession',
                    'first_funding_age']

progression_feats = cohort[progression_cols].rename(
    columns={'id': 'object_id'})

print("\n=== Feature Group 10: Progression ===")
print(f"Shape: {progression_feats.shape}")
print(progression_feats.describe().round(1))


=== Feature Group 10: Progression ===
Shape: (13768, 9)
       reached_angel  reached_series_a  reached_series_b  \
count        13768.0           13768.0           13768.0   
mean             0.2               0.4               0.2   
std              0.4               0.5               0.4   
min              0.0               0.0               0.0   
25%              0.0               0.0               0.0   
50%              0.0               0.0               0.0   
75%              0.0               1.0               0.0   
max              1.0               1.0               1.0   

       reached_series_c_plus  reached_venture  funding_stage_score  \
count                13768.0          13768.0              13768.0   
mean                     0.2              0.4                  2.8   
std                      0.4              0.5                  1.4   
min                      0.0              0.0                  1.0   
25%                      0.0              0.0       

**Funding stage funnel analysis:**
Exit rates increase monotonically with funding stage — from 10.4% for 
angel-stage companies to 22.7% for Series C+ companies (vs 13.9% cohort 
average). This confirms that funding_stage_score is a strong predictor 
of exit outcomes and validates the ordinal encoding chosen.

The count funnel does not show strict sequential decrease (angel < venture) 
due to known under-reporting of informal angel rounds in Crunchbase 
(Menon et al., 2017). Series A through C+ shows the expected decreasing 
pattern (35% → 23.8% → 16.9%).

In [ ]:
# Show the TRUE funnel : how many companies reached EACH stage

stages = ['reached_angel', 'reached_venture', 
          'reached_series_a', 'reached_series_b', 
          'reached_series_c_plus']

print("=== Funding Stage Funnel ===")
print(f"{'Stage':<25} {'Companies':>10} {'% of cohort':>12} {'Exit rate':>10}")
print("-" * 60)
print(f"{'Total cohort':<25} {len(cohort):>10,} {'100.0%':>12} {cohort['label'].mean()*100:>9.1f}%")

for stage in stages:
    subset = cohort[cohort[stage] == 1]
    pct    = len(subset) / len(cohort) * 100
    exits  = subset['label'].mean() * 100
    print(f"{stage:<25} {len(subset):>10,} {pct:>11.1f}% {exits:>9.1f}%")

=== Funding Stage Funnel ===
Stage                      Companies  % of cohort  Exit rate
------------------------------------------------------------
Total cohort                  13,768       100.0%      13.9%
reached_angel                  3,433        24.9%      10.4%
reached_venture                5,790        42.1%      12.5%
reached_series_a               4,822        35.0%      17.5%
reached_series_b               3,271        23.8%      20.5%
reached_series_c_plus          2,326        16.9%      22.7%


### Feature Group 11 — Text Features from Company Descriptions (Lectures 9–10)

Source: objects.csv 

Connects directly to Lecture 9 (tokenisation and embeddings) and 
Lecture 10 (encoder-only transformers for classification).

**Description length features** (Lecture 9 — tokenisation):
- **desc_word_count**: words in description. More words = better 
  documented = higher investor visibility.
- **desc_char_count**: character count. Correlated with word count 
  but captures technical vocabulary depth.
- **has_description**: binary flag. Companies with no description 
  have lower Crunchbase visibility.

**TF-IDF features** (Lecture 9 — text as numeric vectors):

TF-IDF is the standard baseline text representation before neural embeddings,
introduced by Spärck Jones (1972). It converts each company description into 
a numeric vector by scoring each word on two criteria:

- **Term Frequency (TF)**: how often the word appears in THIS company's 
  description — words used repeatedly are important to that company
- **Inverse Document Frequency (IDF)**: how rare the word is across ALL 
  13,768 descriptions — rare words carry more distinguishing information 
  than common words like "company" or "service"

The final TF-IDF score = TF × IDF. This produce a numeric vector representation of each company's text, where each dimension corresponds to one term's importance.

**Leakage prevention:** Outcome-describing terms ('acquired', 'merger', 
'ipo', 'http') are removed , therefore some descriptions were updated post-exit 
and would otherwise leak the label directly.


In [ ]:

# ── Step 1: Description length features (Lecture 9: tokenisation) ─
# Word/character count = simplest tokenisation-derived signal
# More words = better documented = higher investor visibility
cohort['desc_word_count'] = (cohort['overview']
                              .fillna('')
                              .apply(lambda x: len(x.split())))
cohort['desc_char_count'] = (cohort['overview']
                              .fillna('')
                              .apply(len))
cohort['has_description'] = (cohort['overview'].notna()).astype(int)

print("=== Description length features ===")
print(f"Has description: {cohort['has_description'].sum():,} / {len(cohort):,}")
print(cohort[['desc_word_count','desc_char_count']].describe().round(1))

# ── Step 2: Build text column with fallback chain ─────────────

cohort['text'] = cohort['overview'].fillna(
    cohort['description'].fillna(
        cohort['short_description'].fillna('')))

# ── Step 3: Define stop words ─────────────────────────────────
# Standard English stop words + outcome words that cause leakage
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

outcome_words = [
    'acquired', 'acquisition', 'ipo', 'public', 'listed',
    'merger', 'merged', 'purchased', 'bought', 'sold',
    'facebook', 'google', 'microsoft', 'apple', 'twitter',
    'announced', 'completed', 'raised', 'funded',
    'http', 'www', 'com', 'million', 'billion'
]
custom_stop_words = list(ENGLISH_STOP_WORDS) + outcome_words

# ── Step 4: Build TF-IDF features ────────────────────────────
# TF-IDF = Term Frequency × Inverse Document Frequency
# max_features=150  → top 150 most informative terms
# min_df=5          → term must appear in ≥5 companies (removes noise)
# ngram_range=(1,2) → unigrams + bigrams (e.g. "mobile app" as one feature)
# token_pattern     → letters only, min 3 chars (removes numbers, URLs)
tfidf = TfidfVectorizer(
    max_features=150,
    min_df=5,
    stop_words=custom_stop_words,
    ngram_range=(1,2),
    token_pattern=r'[a-zA-Z]{3,}'
)

tfidf_matrix = tfidf.fit_transform(cohort['text'])

# Convert sparse matrix to DataFrame with readable column names
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=[f'tfidf_{t}' for t in tfidf.get_feature_names_out()],
    index=cohort.index
)
tfidf_df['object_id'] = cohort['id'].values

# ── Step 5: Add description length features to same table ─────
tfidf_df['desc_word_count'] = cohort['desc_word_count'].values
tfidf_df['desc_char_count'] = cohort['desc_char_count'].values
tfidf_df['has_description'] = cohort['has_description'].values

# ── Final output ──────────────────────────────────────────────
print("\n=== Feature Group 11: TF-IDF Text (leakage-safe) ===")
print(f"Shape: {tfidf_df.shape}")
print(f"Example features: {list(tfidf_df.columns[:8])}")
print(f"Text coverage: {(cohort['text']!='').sum():,} / {len(cohort):,}")

=== Description length features ===
Has description: 13,747 / 13,768
       desc_word_count  desc_char_count
count          13768.0          13768.0
mean              98.5            683.7
std               76.2            515.4
min                0.0              0.0
25%               52.0            361.0
50%               82.0            570.5
75%              124.0            866.0
max             2066.0          12307.0

=== Feature Group 11: TF-IDF Text (leakage-safe) ===
Shape: (13768, 154)
Example features: ['tfidf_access', 'tfidf_advertising', 'tfidf_allows', 'tfidf_analytics', 'tfidf_application', 'tfidf_applications', 'tfidf_available', 'tfidf_based']
Text coverage: 13,757 / 13,768


In [56]:
# Show TF-IDF output for 5 real companies
sample_idx = cohort.head(5).index

print("=== Company descriptions (first 100 chars) ===")
for i in sample_idx:
    name         = cohort.loc[i, 'name']
    text         = cohort.loc[i, 'overview']
    text_preview = str(text)[:100] if pd.notna(text) else 'NO DESCRIPTION'
    print(f"\n{name}: {text_preview}...")

print("\n=== TF-IDF top 5 features per company ===")
tfidf_sample = tfidf_df[tfidf_df['object_id'].isin(
    cohort.head(5)['id'].values)].copy()

tfidf_cols = [c for c in tfidf_df.columns if c.startswith('tfidf_')]

for i, row in tfidf_sample.iterrows():
    company_name = cohort[cohort['id'] == row['object_id']]['name'].values[0]
    scores       = row[tfidf_cols]
    top5         = scores[scores > 0].sort_values(ascending=False).head(5)
    print(f"\n{company_name}:")
    if len(top5) > 0:
        for feat, val in top5.items():
            print(f"  {feat}: {val:.3f}")
    else:
        print("  (no description — all zeros)")

=== Company descriptions (first 100 chars) ===

Wetpaint: Wetpaint is a technology platform company that uses its proprietary state-of-the-art technology and ...

FriendFeed: [FriendFeed](http://www.friendfeed.com) aims to be a one stop shop for all your social networking up...

Mobclix: Mobclix (www.mobclix.com) is the industry's largest mobile ad exchange network via its sophisticated...

Fitbit: Fitbit inspires people to exercise more, eat better and live
healthier lifestyles. 

The company is ...

MTPV: MTPV Corporation is a clean energy company focus on innovations in Micron-gap ThermalPhotoVoltaics t...

=== TF-IDF top 5 features per company ===

Wetpaint:
  tfidf_media: 0.430
  tfidf_platform: 0.346
  tfidf_technology: 0.323
  tfidf_offices: 0.252
  tfidf_partners: 0.250

FriendFeed:
  tfidf_product: 0.577
  tfidf_social: 0.428
  tfidf_company: 0.382
  tfidf_networks: 0.328
  tfidf_network: 0.280

Mobclix:
  tfidf_application: 0.598
  tfidf_platform: 0.355
  tfidf_mobile: 0.300


### Final Feature Matrix — Merging All Groups

All feature groups are merged onto the cohort using left joins on object_id.
Missing values are filled with 0 after merging.

Final matrix saved to data/processed/features.csv for notebook 3.

In [57]:
# ═══════════════════════════════════════════════════════════════
# FINAL MERGE — All Feature Groups 1–11
# Base: cohort (13,768 companies)
# Join: each feature group on object_id (left join)
# Fill: missing values → 0 after all merges
# ═══════════════════════════════════════════════════════════════

# Start with cohort base columns
features = cohort[['id', 'label', 'founded_year',
                   'country_code', 'category_code',
                   'company_age', 'is_us']].copy()
features = features.rename(columns={'id': 'object_id'})

# Group 1 — funding history
features = features.merge(
    funding_features[['object_id', 'n_rounds',
                      'total_funding_usd', 'avg_round_size',
                      'funding_span_months', 'log_total_funding',
                      'log_avg_round']],
    on='object_id', how='left')

# Group 2 — round types
features = features.merge(
    round_type_feats, on='object_id', how='left')

# Group 4 — investors
features = features.merge(
    investor_features, on='object_id', how='left')

# Group 5 — milestones
features = features.merge(
    milestone_feats, on='object_id', how='left')

# Group 6 — offices
features = features.merge(
    office_feats, on='object_id', how='left')

# Group 7 — team
features = features.merge(
    team_feats, on='object_id', how='left')

# Group 8 — education
features = features.merge(
    edu_feats, on='object_id', how='left')

# Group 9 — funding recency and velocity
features = features.merge(
    recency_feats, on='object_id', how='left')

# Group 10 — round progression and time signals
features = features.merge(
    progression_feats, on='object_id', how='left')

# Group 11 — TF-IDF text + description length
features = features.merge(
    tfidf_df, on='object_id', how='left')

# Sector dummies — aligned by cohort index
sector_dummies_aligned = sector_dummies.copy()
sector_dummies_aligned.index = cohort.index
features = pd.concat([features.reset_index(drop=True),
                      sector_dummies_aligned.reset_index(drop=True)],
                     axis=1)

# Fill missing numeric values with 0
# Absence from table = zero activity, not unknown value
numeric_cols = features.select_dtypes(include=[np.number]).columns
features[numeric_cols] = features[numeric_cols].fillna(0)

# Fill missing categorical values with 'unknown'
features['country_code']  = features['country_code'].fillna('unknown')
features['category_code'] = features['category_code'].fillna('unknown')

# ── Sanity check ──────────────────────────────────────────────
print("=== FINAL FEATURE MATRIX ===")
print(f"Shape: {features.shape}")
print(f"\nLabel distribution:")
print(features['label'].value_counts())
print(f"\nMissing values remaining:")
missing = features.isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print("  None — all clean")
else:
    print(missing)

# ── Save ──────────────────────────────────────────────────────
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(exist_ok=True)
features.to_csv(DATA_PROCESSED / 'features.csv', index=False)
print(f"\nSaved to data/processed/features.csv")
print(f"Total columns: {features.shape[1]}")
print(f"Total features: {features.shape[1] - 5}")
print("(excluding object_id, label, country_code, "
      "category_code, founded_year)")

=== FINAL FEATURE MATRIX ===
Shape: (13768, 221)

Label distribution:
label
0    11850
1     1918
Name: count, dtype: int64

Missing values remaining:
  None — all clean

Saved to data/processed/features.csv
Total columns: 221
Total features: 216
(excluding object_id, label, country_code, category_code, founded_year)


In [58]:
# Show feature count by group
groups = {
    'Base (age, is_us)': ['company_age', 'is_us'],
    'Group 1 (funding history)': ['n_rounds','total_funding_usd',
                                   'avg_round_size','funding_span_months',
                                   'log_total_funding','log_avg_round'],
    'Group 2 (round types)': [c for c in features.columns 
                               if 'funding_round_type' in c],
    'Group 4 (investors)': ['n_investors'],
    'Group 5 (milestones)': ['n_milestones'],
    'Group 6 (offices)': ['n_offices','has_international_office'],
    'Group 7 (team)': ['n_team_total','n_team_current'],
    'Group 8 (education)': ['has_mba','has_phd','has_top_uni'],
    'Group 9 (recency)': ['days_since_last_funding',
                           'avg_days_between_rounds','round_size_growth',
                           'last_round_size_usd','log_last_round_size'],
    'Group 10 (progression)': [c for c in features.columns 
                                if 'reached' in c or 
                                c in ['funding_stage_score',
                                      'founded_in_recession',
                                      'first_funding_age']],
    'Group 11 (text)': [c for c in features.columns 
                        if 'tfidf_' in c or 
                        c in ['desc_word_count','desc_char_count',
                              'has_description']],
    'Sectors': [c for c in features.columns if 'sector_' in c],
}

total = 0
for group, cols in groups.items():
    existing = [c for c in cols if c in features.columns]
    print(f"{group:<35} {len(existing):>3} features")
    total += len(existing)

print(f"\n{'Total counted:':<35} {total:>3}")
print(f"{'Actual features:':<35} {features.shape[1]-5:>3}")

Base (age, is_us)                     2 features
Group 1 (funding history)             6 features
Group 2 (round types)                 9 features
Group 4 (investors)                   1 features
Group 5 (milestones)                  1 features
Group 6 (offices)                     2 features
Group 7 (team)                        2 features
Group 8 (education)                   3 features
Group 9 (recency)                     5 features
Group 10 (progression)                8 features
Group 11 (text)                     153 features
Sectors                              24 features

Total counted:                      216
Actual features:                    216
